In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import MinMaxScaler
import joblib

In [ ]:
# Load dataset
df = pd.read_csv('bitcoin_history.csv')

In [ ]:
# Data Preprocessing
df['Date'] = pd.to_datetime(df['Date'])
numeric_cols = ['Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume']
for col in numeric_cols:
    df[col] = df[col].str.replace(',', '').astype(float)

df = df.sort_values('Date')

In [ ]:
# Feature Scaling
scaler = MinMaxScaler()
df[numeric_cols] = scaler.fit_transform(df[numeric_cols])
joblib.dump(scaler, 'scaler.pkl')  # Save scaler for future predictions

In [ ]:
# Creating Sequential Data for LSTM & Transformer
sequence_length = 30  # Lookback period
X, y = [], []
for i in range(len(df) - sequence_length - 1):
    X.append(df[numeric_cols].iloc[i:i + sequence_length].values)
    y.append(df['Close'].iloc[i + sequence_length])

X, y = np.array(X), np.array(y)
X_tensor = torch.tensor(X, dtype=torch.float32)
y_tensor = torch.tensor(y, dtype=torch.float32)

dataset = TensorDataset(X_tensor, y_tensor)
dataloader = DataLoader(dataset, batch_size=64, shuffle=True)

In [ ]:
# Define "GNN" Layer 
class GNNBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(GNNBlock, self).__init__()
        self.conv1 = nn.Conv1d(in_channels, out_channels, kernel_size=3, padding=1)
        self.conv2 = nn.Conv1d(out_channels, out_channels, kernel_size=3, padding=1)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = x.permute(0, 2, 1)  # Reshape for GNN (batch, features, time)
        x = self.relu(self.conv1(x))
        x = self.relu(self.conv2(x))
        x = x.permute(0, 2, 1)  # Reshape back
        return x

In [ ]:
# Hybrid Model: "GNN" + LSTM + Transformer
class HybridModel(nn.Module):
    def __init__(self, input_dim, cnn_out, lstm_hidden, transformer_heads):
        super(HybridModel, self).__init__()
        
        # "GNN" Layer
        self.gnn = GNNBlock(input_dim, cnn_out)
        
        # LSTM Layer
        self.lstm = nn.LSTM(input_size=cnn_out, hidden_size=lstm_hidden, batch_first=True)
        
        # Transformer Layer
        self.transformer = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(d_model=lstm_hidden, nhead=transformer_heads), num_layers=2)
        
        # Fully Connected Layer
        self.fc = nn.Linear(lstm_hidden, 1)

    def forward(self, x):
        x = self.gnn(x)
        x, _ = self.lstm(x)
        x = self.transformer(x)
        x = self.fc(x[:, -1, :])  # Get last output from LSTM sequence
        return x

In [ ]:
# Initialize Model
model = HybridModel(input_dim=len(numeric_cols), cnn_out=32, lstm_hidden=64, transformer_heads=4)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

In [ ]:
# Loss and Optimizer
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [ ]:
# Training Loop
epochs = 50
for epoch in range(epochs):
    model.train()
    total_loss = 0
    for X_batch, y_batch in dataloader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        
        output = model(X_batch)
        loss = criterion(output.squeeze(), y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    
    print(f'Epoch {epoch+1}/{epochs}, Loss: {total_loss/len(dataloader)}')

# Save trained model
torch.save(model.state_dict(), 'bitcoin_gnn_lstm_transformer.pth')
print('Model saved Sucessfull')

e:\Softwares\anaconda3\envs\kaggle\lib\site-packages\torch\nn\modules\transformer.py:385: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(


Epoch 1/50, Loss: 0.24880359852686523
Epoch 2/50, Loss: 0.007593758908721308
Epoch 3/50, Loss: 0.004617434844840318
Epoch 4/50, Loss: 0.0034794722926259664
Epoch 5/50, Loss: 0.002383533024112694
Epoch 6/50, Loss: 0.0018997129217799133
Epoch 7/50, Loss: 0.001469437055736004
Epoch 8/50, Loss: 0.0012229332802235148
Epoch 9/50, Loss: 0.0011796105420216918
Epoch 10/50, Loss: 0.0010348611736844759
Epoch 11/50, Loss: 0.0011103661236120388
Epoch 12/50, Loss: 0.0009278401004849002
Epoch 13/50, Loss: 0.001044933181644107
Epoch 14/50, Loss: 0.0009184033862159898
Epoch 15/50, Loss: 0.0017581956592039206
Epoch 16/50, Loss: 0.0011768178209119165
Epoch 17/50, Loss: 0.0017917998930594573
Epoch 18/50, Loss: 0.000905857550969813
Epoch 19/50, Loss: 0.0007409328842186369
Epoch 20/50, Loss: 0.0006824585155603321
Epoch 21/50, Loss: 0.0007511591153161135
Epoch 22/50, Loss: 0.0006802759402489755
Epoch 23/50, Loss: 0.0006564508201942469
Epoch 24/50, Loss: 0.0006890557449272213
Epoch 25/50, Loss: 0.000719117633